# Text classification transformer written by Gemini

In [ ]:
import torch
import torch.nn as nn
import math
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer


In [ ]:

# ==========================================
# 1. Positional Encoding
# ==========================================
class PositionalEncoding(nn.Module):
    """
    Injects information about the relative or absolute position of the 
    tokens in the sequence since the Transformer has no inherent notion of order.
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # Create a matrix of [max_len, d_model] representing the positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Calculate the positional encodings (sine for even indices, cosine for odd)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as a buffer so it's not considered a model parameter (doesn't require gradients)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # Add the positional encoding to the input embeddings
        x = x + self.pe[:, :x.size(1), :]
        return x

In [ ]:

# ==========================================
# 2. The Transformer Model
# ==========================================
class TextClassificationTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=512, num_classes=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Embedding layer: turns token IDs into vectors
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # PyTorch's built-in Transformer Encoder Block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout,
            batch_first=True # We want inputs of shape (batch_size, sequence_length, embedding_dim)
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Final classification head
        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        # x shape: (batch_size, seq_len)
        
        # 1. Embed the tokens and scale them
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        # 2. Add positional encoding
        x = self.pos_encoder(x)
        
        # 3. Pass through the Transformer
        # We use src_key_padding_mask to ignore the <PAD> tokens during attention
        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        
        # 4. Pooling: Take the output of the first token (often treated as the [CLS] token) 
        # Alternatively, we can take the mean of all sequence tokens. We'll use mean pooling here.
        # Mask out padding tokens before taking the mean
        if padding_mask is not None:
            mask = (~padding_mask).unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            x = x.mean(dim=1)
            
        x = self.dropout(x)
        
        # 5. Classify
        logits = self.classifier(x)
        return logits


In [ ]:

# ==========================================
# 3. Data Preparation (IMDB Dataset)
# ==========================================
def prepare_data(batch_size=32, max_length=256):
    print("Loading IMDB dataset and tokenizer...")
    # Load dataset
    dataset = load_dataset("imdb")
    
    # Use a pre-trained basic tokenizer (like distilbert) to quickly get a vocabulary
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)
    
    print("Tokenizing data...")
    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    
    train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=batch_size)
    test_dataloader = DataLoader(tokenized_datasets["test"], batch_size=batch_size)
    
    return train_dataloader, test_dataloader, tokenizer.vocab_size


In [ ]:

# ==========================================
# 4. Training Loop
# ==========================================
def train_model():
    # Hyperparameters
    batch_size = 32
    max_length = 256
    epochs = 3
    learning_rate = 1e-4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Setup Data
    train_loader, test_loader, vocab_size = prepare_data(batch_size, max_length)

    # Initialize Model, Loss, and Optimizer
    model = TextClassificationTransformer(
        vocab_size=vocab_size,
        d_model=128,      # Kept small for fast local training
        nhead=4,
        num_layers=2,
        dim_feedforward=256
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Training
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for i, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            # attention_mask is 1 for real tokens, 0 for padding. 
            # PyTorch's padding_mask expects True for padding tokens to ignore them.
            padding_mask = (batch['attention_mask'] == 0).to(device) 
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(input_ids, padding_mask=padding_mask)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            if i % 100 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{i}/{len(train_loader)}], Loss: {loss.item():.4f}")
                
        print(f"--- Epoch {epoch+1} completed. Average Loss: {total_loss/len(train_loader):.4f} ---")

if __name__ == "__main__":
    # Run the training loop (Warning: Training takes time without a GPU!)
    train_model()